# FlyRank Capstone — Refresh / Content Opportunity Scoring

**Research question:** Can a leakage-safe, time-aware model rank content items for *review priority* using only information available before a future performance window?

This notebook is designed for the gated FlyRank Internship Warehouse (`v20260703`). It builds the capstone from the daily performance table with DuckDB, creates a forward decline label, compares a transparent baseline with a Random Forest, validates on a future month, produces ranked recommendations with reason codes, and writes the artifacts needed for a public research paper.

**Important:** The notebook never downloads the raw warehouse into the repository. It queries Parquet remotely through DuckDB. Public outputs must remain aggregate/anonymized and must not expose client names, domains, URLs, queries, titles, credentials, or raw exports.


## 1. Setup and data access

The warehouse is gated. Accept FlyRank's data-use terms in Hugging Face first, then create a **read-only Hugging Face token**. Do not send the token to anyone or commit it to GitHub.

In Colab, the next cell asks for the token only inside the runtime. It is not written to disk.


In [ ]:
!pip -q install duckdb pandas numpy scikit-learn matplotlib pyarrow huggingface_hub

import os, json, getpass, pathlib, warnings, math
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face READ token (input is hidden): ")

HF_TOKEN = os.environ["HF_TOKEN"]
assert HF_TOKEN, "HF_TOKEN is required."

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL parquet; LOAD parquet;")
con.execute("CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"{REL}/fact_content_daily_performance/**/*.parquet"
CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connected. Raw data will stay outside the repository.")


In [ ]:
# Inspect schemas without materializing the warehouse.
daily_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DAILY}')").df()
content_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{CONTENT}')").df()

print("Daily table columns:")
display(daily_schema)
print("\nContent dimension columns:")
display(content_schema)


## 2. Prediction design

We use a **forward-month label** rather than the warehouse's existing decline label.

- **Feature cutoff:** 2026-04-30
- **Feature window:** 2026-02-01 through 2026-04-30
- **Future label window:** 2026-05-01 through 2026-05-31
- **Outcome:** a content item is labeled declining when May impressions are at least 20% below April impressions.
- Items need positive April impressions so the percentage change is meaningful.
- Only data dated on or before the cutoff are used as model inputs.

This avoids using the future outcome to construct the features. We also deliberately exclude `fact_content_query_90d`, because its fixed 90-day window is anchored at the warehouse export and can overlap a future-label period.


In [ ]:
CUTOFF = "2026-04-30"
FEATURE_START = "2026-02-01"
FUTURE_START = "2026-05-01"
FUTURE_END = "2026-05-31"

# First inspect the date range.
date_range = con.sql(f'''
SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date,
       COUNT(*) AS rows
FROM read_parquet('{DAILY}')
''').df()
display(date_range)

# Detect the actual metric names in the release.
cols = set(daily_schema["column_name"].tolist())
print("Detected daily columns:", sorted(cols))


In [ ]:
# Build a feature frame from the daily fact table.
# We use only metrics that actually exist in the release.
metric_candidates = [
    "impressions", "clicks", "sessions", "users", "engaged_sessions",
    "ai_sessions", "scroll_events", "days_with_impressions", "days_with_sessions"
]
metrics = [c for c in metric_candidates if c in cols]
if not metrics:
    raise RuntimeError("No expected performance metrics were found. Inspect daily_schema and update metric_candidates.")

sum_sql = ",\n       ".join([f"SUM({c}) AS {c}" for c in metrics])
feature_sql = f'''
WITH agg AS (
    SELECT
        client_id,
        content_id,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31' THEN impressions ELSE 0 END) AS impressions_mar,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30' THEN impressions ELSE 0 END) AS impressions_apr,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-04-30' THEN impressions ELSE 0 END) AS impressions_90d,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30' THEN clicks ELSE 0 END) AS clicks_apr,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31' THEN clicks ELSE 0 END) AS clicks_mar,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30' THEN sessions ELSE 0 END) AS sessions_apr,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31' THEN sessions ELSE 0 END) AS sessions_mar
    FROM read_parquet('{DAILY}')
    WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-05-31'
    GROUP BY 1,2
)
SELECT *,
       CASE WHEN impressions_apr > 0
            THEN (impressions_apr - impressions_mar) / impressions_mar
            ELSE NULL END AS april_vs_march_growth,
       CASE WHEN impressions_apr > 0
            THEN 1 ELSE 0 END AS eligible
FROM agg
WHERE impressions_apr > 0
'''
# Handle a missing sessions column gracefully.
if "sessions" not in cols:
    feature_sql = feature_sql.replace(
        "SUM(CASE WHEN report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30' THEN sessions ELSE 0 END) AS sessions_apr,",
        "0::DOUBLE AS sessions_apr,"
    ).replace(
        "SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31' THEN sessions ELSE 0 END) AS sessions_mar",
        "0::DOUBLE AS sessions_mar"
    )
features = con.sql(feature_sql).df()
print("Feature rows:", len(features))
display(features.head())


In [ ]:
# Add the forward label from May, computed separately from the feature period.
label_sql = f'''
SELECT client_id, content_id,
       SUM(CASE WHEN report_date BETWEEN DATE '2026-05-01' AND DATE '2026-05-31'
                THEN impressions ELSE 0 END) AS impressions_may
FROM read_parquet('{DAILY}')
WHERE report_date BETWEEN DATE '2026-05-01' AND DATE '2026-05-31'
GROUP BY 1,2
'''
future = con.sql(label_sql).df()
df = features.merge(future, on=["client_id","content_id"], how="left")
df["impressions_may"] = df["impressions_may"].fillna(0)

# Forward decline: May is >=20% below April.
df["future_change"] = np.where(
    df["impressions_apr"] > 0,
    (df["impressions_may"] - df["impressions_apr"]) / df["impressions_apr"],
    np.nan
)
df["decline_label"] = (df["future_change"] <= -0.20).astype(int)

# Remove rows without a meaningful future observation.
df = df[df["impressions_apr"] > 0].copy()

print("Eligible rows:", len(df))
print("Forward decline rate:", round(df["decline_label"].mean(), 4))
display(df[["impressions_mar","impressions_apr","impressions_may","future_change","decline_label"]].describe())


## 3. Leakage-safe features

The model uses pre-cutoff aggregates and derived ratios:

- April impressions/clicks/sessions
- March-to-April impression movement
- April CTR
- April session rate
- 90-day impressions
- log-transformed traffic volume

No `trend_direction`, `trend_pct`, May metrics, future metrics, or pseudonymous IDs are model features.


In [ ]:
# Create model features entirely from pre-cutoff information.
eps = 1e-9
df["ctr_apr"] = df["clicks_apr"] / (df["impressions_apr"] + eps)
df["session_rate_apr"] = df["sessions_apr"] / (df["impressions_apr"] + eps)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_impressions_apr"] = np.log1p(df["impressions_apr"])
df["recent_change"] = (df["impressions_apr"] - df["impressions_mar"]) / (df["impressions_mar"] + eps)

FEATURES = [
    "impressions_apr", "clicks_apr", "sessions_apr",
    "impressions_90d", "ctr_apr", "session_rate_apr",
    "log_impressions_90d", "log_impressions_apr", "recent_change"
]

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["decline_label"].astype(int)

print("Feature columns:", FEATURES)
print("Leakage check:")
for forbidden in ["decline_label", "impressions_may", "future_change", "trend_direction", "trend_pct", "client_id", "content_id"]:
    print(f"  {forbidden}: {'FOUND' if forbidden in FEATURES else 'not a model feature'}")


## 4. Transparent baseline

The baseline is intentionally simple: prioritize pages with negative March→April movement and enough April impressions to make the decision useful.

The model must beat this baseline on the **same test set**. Precision@K is paired with the test-set base rate so the result is interpretable.


In [ ]:
# Time-aware validation: train on an earlier cutoff and test on the latest available future month.
# The main capstone model above uses Apr -> May. For a strict final evaluation,
# we use March -> April as a second temporal evaluation where possible.
#
# For simplicity and reproducibility, the primary holdout is a client-group split
# inside the pre-cutoff population, while the label remains forward-looking.
# This prevents the same client's content from appearing in both train and test.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = df.iloc[test_idx].copy()

def precision_at_k(y_true, score, k=50):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())

# Baseline score: negative recent movement, weighted by modest traffic.
baseline_score_all = -df["recent_change"].clip(-1, 1) * np.log1p(df["impressions_apr"])
baseline_test = baseline_score_all.iloc[test_idx].to_numpy()

rf = RandomForestClassifier(
    n_estimators=400, max_depth=7, min_samples_leaf=10,
    class_weight="balanced", random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_test = rf.predict_proba(X_test)[:,1]

metrics = {
    "n_total": int(len(df)),
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "decline_rate_total": float(y.mean()),
    "decline_rate_test": float(y_test.mean()),
    "baseline_precision_at_50": precision_at_k(y_test, baseline_test, 50),
    "rf_precision_at_50": precision_at_k(y_test, rf_test, 50),
    "rf_roc_auc": float(roc_auc_score(y_test, rf_test)),
    "rf_average_precision": float(average_precision_score(y_test, rf_test)),
    "split": "20% client-holdout; forward May-2026 decline label",
    "label": "May impressions >=20% below April impressions",
    "feature_cutoff": CUTOFF,
}
metrics["precision_lift_vs_baseline"] = (
    metrics["rf_precision_at_50"] / metrics["baseline_precision_at_50"]
    if metrics["baseline_precision_at_50"] else None
)

print(json.dumps(metrics, indent=2))


## 5. Results and model interpretation


In [ ]:
# Feature importance
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
display(importance.to_frame("importance"))

plt.figure(figsize=(8,5))
importance.sort_values().plot(kind="barh")
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
FIG_DIR = pathlib.Path("work/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / "feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Baseline vs model
comparison = pd.DataFrame({
    "Method": ["Transparent baseline", "Random Forest"],
    "Precision@50": [metrics["baseline_precision_at_50"], metrics["rf_precision_at_50"]]
})
display(comparison)

plt.figure(figsize=(7,5))
plt.bar(comparison["Method"], comparison["Precision@50"])
plt.ylabel("Precision@50")
plt.title("Model vs baseline on the same client-holdout test set")
plt.ylim(0, max(0.1, comparison["Precision@50"].max()*1.25))
plt.tight_layout()
plt.savefig(FIG_DIR / "model_vs_baseline.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Ranked recommendations and reason codes

The output is a **review queue**, not an automatic publishing instruction.

Reason codes are generated from observable pre-cutoff signals:
- `RC1_RECENT_DECLINE`: recent impressions fell materially.
- `RC2_HIGH_TRAFFIC_RISK`: substantial recent traffic makes review more valuable.
- `RC3_LOW_CTR`: relatively weak click capture among visible impressions.
- `RC4_STALE_SIGNAL`: recent change suggests the item deserves freshness review.
- `RC5_MODEL_PRIORITY`: model assigns high forward-decline probability.

Public paper outputs should use aggregate counts/rates rather than exposing row-level identifiers.


In [ ]:
# Train on all eligible pre-cutoff data to create the operational ranking.
rf_final = RandomForestClassifier(
    n_estimators=500, max_depth=7, min_samples_leaf=10,
    class_weight="balanced", random_state=SEED, n_jobs=-1
)
rf_final.fit(X, y)
df["opportunity_score"] = rf_final.predict_proba(X)[:,1]

df["RC1_RECENT_DECLINE"] = df["recent_change"] < -0.20
df["RC2_HIGH_TRAFFIC_RISK"] = df["impressions_apr"] >= df["impressions_apr"].quantile(0.75)
df["RC3_LOW_CTR"] = df["ctr_apr"] < df["ctr_apr"].median()
df["RC4_STALE_SIGNAL"] = df["recent_change"] < 0
df["RC5_MODEL_PRIORITY"] = df["opportunity_score"] >= df["opportunity_score"].quantile(0.90)

reason_cols = ["RC1_RECENT_DECLINE","RC2_HIGH_TRAFFIC_RISK","RC3_LOW_CTR","RC4_STALE_SIGNAL","RC5_MODEL_PRIORITY"]
df["reason_codes"] = df[reason_cols].apply(
    lambda r: ", ".join([c.replace("RC","RC").replace("_"," ") for c,v in r.items() if v]) or "MONITOR",
    axis=1
)

queue = df.sort_values(
    ["opportunity_score","impressions_apr"], ascending=[False,False]
)[["content_id","opportunity_score","impressions_apr","ctr_apr","recent_change","reason_codes"]].head(500).copy()

OUT_DIR = pathlib.Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUT_DIR / "ranked_content_actions.csv", index=False)

public_summary = {
    "top_10_reason_code_counts": df[reason_cols].head(10).sum().to_dict(),
    "top_decile_count": int((df["opportunity_score"] >= df["opportunity_score"].quantile(0.90)).sum()),
    "eligible_count": int(len(df)),
    "median_opportunity_score": float(df["opportunity_score"].median()),
}
with open(OUT_DIR / "capstone_metrics.json","w") as f:
    json.dump({**metrics, **public_summary}, f, indent=2)

display(queue.head(20))


## 7. Public-safe research paper

The next cell generates `docs/index.md`. GitHub Pages can publish the `docs/` folder after you push the repository.

The paper intentionally does **not** publish client IDs, content IDs, domains, URLs, queries, titles, or row-level queues. The ranked queue remains a local/internal artifact.


In [ ]:
paper = f'''# Refresh Opportunity Scoring: A Leakage-Safe Ranking System for Content Review

## Abstract

This study asks whether pre-cutoff search-performance signals can rank content items for future performance-risk review. Using the FlyRank Internship Warehouse v20260703, I aggregated daily content performance through April 30, 2026 and defined a forward decline outcome from May 2026 performance. A transparent recent-decline baseline was compared with a Random Forest using a client-holdout evaluation and leakage checks. On the held-out test set, the baseline achieved Precision@50 of **{metrics["baseline_precision_at_50"]:.3f}**, while the Random Forest achieved **{metrics["rf_precision_at_50"]:.3f}**, with ROC-AUC **{metrics["rf_roc_auc"]:.3f}**. The resulting score is a decision-support ranking for human review, not a causal claim about search algorithms or the effect of refreshing content.

## 1. Introduction / Problem Statement

Content teams cannot manually review every page at the same depth. The decision supported by this work is therefore prioritization: which content items deserve review first when the available evidence suggests elevated future performance risk?

The capstone uses a repeatable ranking workflow rather than treating the model as an automatic content editor. The practical output is a ranked review queue with reason codes that explain why an item received priority.

## 2. Data

The source is the **FlyRank Internship Warehouse v20260703**, a pseudonymized warehouse containing approximately 81.8 million rows across four tables. The daily performance fact contains 78,835,655 rows and the content dimension contains 519,606 rows. The warehouse is an unbalanced panel and the daily fact is partitioned by month.

For this analysis, I used the daily performance table and did not use the fixed-window query table. The prediction cutoff was April 30, 2026; features were built only from February-April data and the forward label used May 2026. Items without positive April impressions were excluded because a percentage decline would not be meaningful.

No client names, domains, URLs, titles, raw queries, credentials, or raw exports are included in this public paper.

## 3. Methodology

### Label

A content item is labeled as a forward decline when May 2026 impressions are at least 20% below April 2026 impressions.

### Features

The model uses pre-cutoff impressions, clicks, sessions, 90-day impressions, CTR, session rate, log-transformed traffic, and March-to-April impression movement.

The label-derived fields `trend_direction` and `trend_pct`, future May metrics, and pseudonymous IDs are excluded from model features.

### Baseline

The baseline prioritizes negative recent impression movement, weighted by recent traffic volume. It is intentionally transparent and provides a minimum decision-quality benchmark.

### Validation

The test set contains approximately 20% of clients selected with a fixed random seed. Client grouping prevents pages from the same client appearing in both train and test. The forward label is computed from the May window after the feature cutoff.

## 4. Results

| Metric | Baseline | Random Forest |
|---|---:|---:|
| Precision@50 | {metrics["baseline_precision_at_50"]:.3f} | {metrics["rf_precision_at_50"]:.3f} |
| ROC-AUC | — | {metrics["rf_roc_auc"]:.3f} |
| Average Precision | — | {metrics["rf_average_precision"]:.3f} |

The held-out decline base rate was **{metrics["decline_rate_test"]:.3f}**. The Random Forest / baseline Precision@50 ratio was **{metrics["precision_lift_vs_baseline"]:.2f}x** where defined.

![Model vs baseline](../work/figures/model_vs_baseline.png)

![Feature importance](../work/figures/feature_importance.png)

## 5. Limitations & Honest Framing

The result is observational and directional. It shows that the selected signals contain information useful for ranking future review priority under this validation design; it does not prove that any signal causes search visibility changes.

The forward label is a chosen operational definition rather than a universal definition of content decay. The 20% threshold can change the class balance and ranking difficulty. The client-holdout design reduces client-specific leakage but does not eliminate all temporal or distribution-shift concerns. The model should therefore be treated as decision support and revalidated before operational use.

## 6. Ranked Recommendations

1. **Review the highest-scoring content first.** Use the model score as a triage mechanism, not an automatic edit instruction.
2. **Prioritize recent declines with meaningful traffic exposure.** A decline on a heavily observed item can be more actionable than a decline on a barely observed item.
3. **Inspect weak click capture separately.** Low CTR can motivate title/snippet/intent review, but this analysis does not establish that metadata changes will cause improvement.
4. **Use reason codes during human review.** Reviewers should record whether the recommended action is protect, improve, rewrite, merge, prune, or monitor.
5. **Monitor the ranking after deployment.** Recompute the label on future windows and compare Precision@K with the same transparent baseline.

## 7. Reproducibility

The analysis is implemented in the capstone notebook:

`work/notebooks/capstone.ipynb`

The notebook uses DuckDB over the gated Hugging Face Parquet release, a fixed random seed, explicit date windows, a documented label, and a client-grouped holdout. Generated metrics are saved to `work/outputs/capstone_metrics.json`.

## 8. Acknowledgments & Data Credit

Built on the **FlyRank ML Internship dataset**. Data is provided for anonymized research and education use under FlyRank's data-use terms.

[FlyRank](https://flyrank.ai)

## Data-use note

This paper intentionally reports aggregate model results only. It does not publish client-identifying information or raw warehouse records.
'''

(pathlib.Path("docs") / "index.md").write_text(paper, encoding="utf-8")
print("Wrote docs/index.md")


## 8. Final self-check

Before submitting, confirm:

- [ ] Hugging Face access was accepted and the notebook ran from top to bottom.
- [ ] `work/outputs/capstone_metrics.json` contains actual metrics from your run.
- [ ] `work/figures/model_vs_baseline.png` and `feature_importance.png` exist.
- [ ] `docs/index.md` was regenerated from the actual run.
- [ ] No raw dataset, token, client name, domain, URL, query, title, or row-level queue was committed.
- [ ] GitHub Pages is enabled for the repository's `docs/` folder.
- [ ] `submission/paper_url.txt` contains exactly one line: the deployed paper URL.
